# 02 - Batch Evaluation All Categories

Run full CSMS AI scoring pipeline across all 29 categories with hybrid chunking and HTML export.

In [ ]:
import sys, os, json, time
if os.path.basename(os.getcwd()) == "notebook_v3":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
MARINE_DIR = os.path.join(os.getcwd(), "test_submissions_dedicated", "PT Newport Marine Services")
INFO_FILE = os.path.join(MARINE_DIR, "information.json")

company_info = {}
if os.path.exists(INFO_FILE):
    with open(INFO_FILE, "r", encoding="utf-8") as f:
        company_info = json.load(f)

START_DATE = company_info.get("date_start", None)
END_DATE = company_info.get("date_end", None)
COMPANY = company_info.get("company", None)
CHUNKING_MODE = "memory_only"
PERSIST_FAILED = True
JUROR_ROLE = "reasoning_model"
CONSENSUS_ROLE = "consensus_model"

print(f"Loaded company info: {json.dumps(company_info, indent=2)}")
print(f"Period:        {START_DATE} to {END_DATE}")
print(f"Company:       {COMPANY}")
print(f"Chunking Mode: {CHUNKING_MODE}")
print(f"Marine Dir:    {MARINE_DIR}")

In [ ]:
from engine.pipeline import evaluate_all_categories
from engine.utils.eval_config import EvaluationConfig
from engine.utils.chunking_config import ChunkingMode

config = EvaluationConfig(
    start_date=START_DATE,
    end_date=END_DATE,
    company=COMPANY,
    marine_dir=MARINE_DIR,
    juror_role=JUROR_ROLE,
    consensus_role=CONSENSUS_ROLE,
    chunking_mode=ChunkingMode(CHUNKING_MODE),
    persist_failed=PERSIST_FAILED,
    company_info=company_info
)

print("Evaluation config:")
print(json.dumps(config.to_dict(), indent=2))
print("=" * 80)

start_time = time.time()
report = evaluate_all_categories(config=config)
total_time = time.time() - start_time
print(f"\nTotal batch time: {total_time:.1f}s")

## 2. Discover All Categories

In [ ]:
from engine.pipeline import discover_category_folders

categories = discover_category_folders(MARINE_DIR)
print(f"Found {len(categories)} categories:")
for c in categories:
    fcount = sum(len(f) for _, _, f in os.walk(c["path"]))
    print(f"  {c["category_id"]:>5} | {fcount:>3} files | {c["folder_name"]}")

## 3. Run Batch Evaluation

In [ ]:
from engine.pipeline import evaluate_all_categories
from engine.utils.eval_config import EvaluationConfig
from engine.utils.chunking_config import ChunkingMode

config = EvaluationConfig(
    start_date=START_DATE,
    end_date=END_DATE,
    company=COMPANY,
    marine_dir=MARINE_DIR,
    juror_role=JUROR_ROLE,
    consensus_role=CONSENSUS_ROLE,
    chunking_mode=ChunkingMode(CHUNKING_MODE),
    persist_failed=PERSIST_FAILED,
    company_info=company_info
)

print("Evaluation config:")
print(json.dumps(config.to_dict(), indent=2))
print("=" * 80)

start_time = time.time()
report = evaluate_all_categories(config=config)
total_time = time.time() - start_time
print(f"\nTotal batch time: {total_time:.1f}s")

## 4. Summary Dashboard

In [ ]:
print("=" * 70)
print(f"  BATCH RESULTS")
print("=" * 70)
print(f"  Total Categories: {report["total_categories"]}")
print(f"  Passed:           {report["passed"]}")
print(f"  Needs Review:     {report["needs_review"]}")
print(f"  Failed:           {report["failed"]}")
print(f"  Accuracy:         {report["accuracy"]}%")
print("=" * 70)

## 5. Per-Category Results Table

In [ ]:
header = f"{"Cat":<6} {"Exp":>4} {"Comp":>5} {"Ev":>4} {"HSE":>4} {"Final":>6} {"Conf":>6} {"Match":>6} {"Status":<12} {"Time":>7}"
print(f"
{header}")
print("-" * 75)

for r in report.get("results", []):
    cat = r.get("category_id", "?")
    exp = r.get("expected_score", 0)
    comp = r.get("compliance_score", 0)
    ev = r.get("evidence_score", 0)
    hse = r.get("hse_score", 0)
    final = r.get("final_score", 0)
    conf = r.get("confidence", 0)
    match = "YES" if exp == final else "NO"
    status = r.get("status", "?")
    t = r.get("processing_time_sec", 0)
    print(f"{cat:<6} {exp:>4} {comp:>5} {ev:>4} {hse:>4} {final:>6} {conf:>6.2f} {match:>6} {status:<12} {t:>6.1f}s")

## 6. Batch HTML Viewer

In [ ]:
html_path = report.get("html_viewer_path", "N/A")
print(f"Batch HTML Viewer: {html_path}")

if os.path.exists(html_path):
    print(f"File size: {os.path.getsize(html_path) / 1024:.1f} KB")
    print(f"Open in browser: file:///{os.path.abspath(html_path)}")

## 7. Save Full Report

In [ ]:
os.makedirs("notebook_v3/outputs", exist_ok=True)
out_path = "notebook_v3/outputs/batch_evaluation_results.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print(f"Full report saved to {out_path}")
print(f"File size: {os.path.getsize(out_path) / 1024:.1f} KB")